**Jakub Orchowski, s223281**

# CEL ĆWICZENIA
Punktowe i lokalne metody przetwarzania obrazów

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage.filters import threshold_local, threshold_otsu
from skimage.measure import euler_number as skimage_euler_number
%matplotlib inline


# Zadania

In [2]:
def filter2d(image, kernel):
    """Odpowiednik MATLAB-owej funkcji filter2 (korelacja 2D)."""
    kh, kw = kernel.shape
    pad_h, pad_w = kh // 2, kw // 2
    padded = np.pad(image.astype(np.float64), ((pad_h, pad_h), (pad_w, pad_w)), mode='constant')
    windows = np.lib.stride_tricks.sliding_window_view(padded, (kh, kw))
    return np.einsum('ijkl,kl->ij', windows, kernel)

## Zadanie 1.
Napisz funkcje do binaryzacji obrazu monochromatycznego:
- **(a)** z ustalonym progiem
- **(b)** z progiem wyznaczonym metodą Otsu
- **(c)** adaptacyjna (lokalna) binaryzacja z otoczeniem 21×21

### (a) Binaryzacja z ustalonym progiem

In [3]:
def binarize_fixed(image, threshold):
    """Binaryzacja obrazu monochromatycznego z ustalonym progiem."""
    return (image >= threshold).astype(np.uint8) * 255

### (b) Binaryzacja metodą Otsu

In [ ]:
def otsu_threshold(image):
    """Wyznaczanie progu binaryzacji metodą Otsu z biblioteki scikit-image."""
    return float(threshold_otsu(image))


def binarize_otsu(image):
    """Binaryzacja obrazu monochromatycznego z progiem Otsu."""
    thresh = otsu_threshold(image)
    print(f'Próg Otsu: {thresh:.2f}')
    return binarize_fixed(image, thresh)

### (c) Adaptacyjna binaryzacja lokalna (otoczenie 21×21)

In [ ]:
def binarize_adaptive(image, block_size=21):
    """Adaptacyjna binaryzacja lokalna z użyciem threshold_local z biblioteki scikit-image."""
    if block_size % 2 == 0:
        raise ValueError('Rozmiar otoczenia musi być nieparzysty.')

    img = image.astype(np.float64)
    local_threshold = threshold_local(img, block_size=block_size, method='mean')
    return (img >= local_threshold).astype(np.uint8) * 255

### Test binaryzacji na obrazach text1.bmp i C.bmp

In [ ]:
Im_text = np.array(Image.open('text1.bmp').convert('L'))
Im_C    = np.array(Image.open('C.bmp').convert('L'))

for name, img in [('text1.bmp', Im_text), ('C.bmp', Im_C)]:
    bin_fixed    = binarize_fixed(img, 128)
    bin_otsu     = binarize_otsu(img)
    bin_adaptive = binarize_adaptive(img)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(img, cmap='gray')
    axes[0].set_title(f'{name} — oryginał')
    axes[0].axis('off')
    axes[1].imshow(bin_fixed, cmap='gray')
    axes[1].set_title('Próg stały (128)')
    axes[1].axis('off')
    axes[2].imshow(bin_otsu, cmap='gray')
    axes[2].set_title('Otsu')
    axes[2].axis('off')
    axes[3].imshow(bin_adaptive, cmap='gray')
    axes[3].set_title('Adaptacyjna (21×21)')
    axes[3].axis('off')
    plt.tight_layout()
    plt.show()

### Wnioski
Binaryzacja z progiem stałym działa dobrze tylko gdy oświetlenie obrazu jest równomierne.

Metoda Otsu automatycznie dobiera próg maksymalizując wariancję międzyklasową — daje lepsze wyniki niż arbitralny próg.

Binaryzacja adaptacyjna radzi sobie najlepiej z nierównomiernym oświetleniem,
ponieważ próg jest wyznaczany lokalnie dla każdego piksela na podstawie jego otoczenia 21×21.

## Zadanie 2.
Napisz funkcję do uśredniania obrazu monochromatycznego operatorami o dowolnych prostokątnych rozmiarach.
Sprawdź dla operatorów 3×3, 7×7 oraz 11×11.

Do realizacji dopuszczalne jest jedynie użycie funkcji filter2 (zaimplementowanej powyżej).

In [ ]:
def average_filter(image, kh, kw):
    """Uśrednianie obrazu operatorem prostokątnym kh×kw."""
    kernel = np.ones((kh, kw), dtype=np.float64) / (kh * kw)
    return filter2d(image, kernel)

In [ ]:
Im_A = np.array(Image.open('A.bmp').convert('L'))
Im_B = np.array(Image.open('B.bmp').convert('L'))
Im_C = np.array(Image.open('C.bmp').convert('L'))

for name, img in [('A.bmp', Im_A), ('B.bmp', Im_B), ('C.bmp', Im_C)]:
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(img, cmap='gray')
    axes[0].set_title(f'{name} — oryginał')
    axes[0].axis('off')

    for idx, k in enumerate([3, 7, 11]):
        result = average_filter(img, k, k)
        axes[idx + 1].imshow(result, cmap='gray')
        axes[idx + 1].set_title(f'Uśrednianie {k}×{k}')
        axes[idx + 1].axis('off')

    plt.tight_layout()
    plt.show()

### Wnioski
Im większy rozmiar operatora uśredniającego, tym silniejsze rozmycie obrazu.

Operator 3×3 lekko wygładza obraz, zachowując większość szczegółów.

Operator 11×11 znacząco rozmywa obraz, usuwając drobne detale i szum,
ale jednocześnie powodując utratę ostrości krawędzi.

## Zadanie 3.
Napisz funkcję do wyznaczania krawędzi w obrazach monochromatycznych za pomocą operatorów Sobela.
Wyświetl: krawędzie poziome, krawędzie pionowe, oraz oba rodzaje krawędzi.

Dodatkowo, analogiczne funkcje dla operatorów Prewitta i Robertsa.

In [ ]:
# Operatory Sobela
SOBEL_H = np.array([[-1, -2, -1],
                     [ 0,  0,  0],
                     [ 1,  2,  1]], dtype=np.float64)

SOBEL_V = np.array([[-1,  0,  1],
                     [-2,  0,  2],
                     [-1,  0,  1]], dtype=np.float64)

# Operatory Prewitta
PREWITT_H = np.array([[-1, -1, -1],
                       [ 0,  0,  0],
                       [ 1,  1,  1]], dtype=np.float64)

PREWITT_V = np.array([[-1,  0,  1],
                       [-1,  0,  1],
                       [-1,  0,  1]], dtype=np.float64)

# Operatory Robertsa
ROBERTS_1 = np.array([[ 0,  1],
                       [-1,  0]], dtype=np.float64)

ROBERTS_2 = np.array([[ 1,  0],
                       [ 0, -1]], dtype=np.float64)


def edge_detect(image, kernel_h, kernel_v):
    """Wyznaczanie krawędzi za pomocą pary operatorów (poziomy i pionowy)."""
    edges_h = filter2d(image, kernel_h)
    edges_v = filter2d(image, kernel_v)
    edges   = np.sqrt(edges_h**2 + edges_v**2)
    return edges_h, edges_v, edges


def normalize(channel):
    lo, hi = channel.min(), channel.max()
    return (channel - lo) / (hi - lo) if hi > lo else np.zeros_like(channel)

In [ ]:
for name, img in [('B.bmp', Im_B), ('C.bmp', Im_C)]:
    for op_name, kh, kv in [('Sobel', SOBEL_H, SOBEL_V),
                             ('Prewitt', PREWITT_H, PREWITT_V),
                             ('Roberts', ROBERTS_1, ROBERTS_2)]:
        eh, ev, e = edge_detect(img, kh, kv)

        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        axes[0].imshow(img, cmap='gray')
        axes[0].set_title(f'{name} — oryginał')
        axes[0].axis('off')
        axes[1].imshow(normalize(eh), cmap='gray')
        axes[1].set_title(f'{op_name} — krawędzie poziome')
        axes[1].axis('off')
        axes[2].imshow(normalize(ev), cmap='gray')
        axes[2].set_title(f'{op_name} — krawędzie pionowe')
        axes[2].axis('off')
        axes[3].imshow(normalize(e), cmap='gray')
        axes[3].set_title(f'{op_name} — oba rodzaje')
        axes[3].axis('off')
        plt.tight_layout()
        plt.show()

### Wnioski
Operator Sobela daje najsilniejszą odpowiedź na krawędzie dzięki wagom 2 przy centralnym pikselu.

Operator Prewitta daje nieco słabszą odpowiedź (jednolite wagi), ale jest prostszy obliczeniowo.

Operator Robertsa używa maski 2×2, więc jest wrażliwszy na szum,
ale lepiej lokalizuje cienkie krawędzie.

Łączenie krawędzi poziomych i pionowych (pierwiastek sumy kwadratów) daje pełny obraz krawędzi.

## Zadanie 4.
Napisz funkcję do wyznaczania liczby Eulera w obrazie binarnym.
Obiekty w obrazie są czarne (wartość 0), tło białe (wartość 255).
Dla czarnych obszarów stosujemy zasadę 4-spójności.

Liczba Eulera: $E = C - H$ (obiektów minus dziur).

Metoda: zliczanie wzorców w oknach 2×2 (bit-quads).

In [ ]:
def euler_number(binary_image):
    """Wyznaczanie liczby Eulera dla obrazu binarnego (obiekty=0, tło=255).
    Odpowiednik MATLAB-owej funkcji bweuler.
    Stosujemy 4-spójność dla obiektów.
    """
    fg = (binary_image == 0).astype(np.uint8)
    return int(skimage_euler_number(fg, connectivity=1))


In [ ]:
Im_bin1 = np.array(Image.open('binary1.bmp').convert('L'))
Im_bin3 = np.array(Image.open('binary3.bmp').convert('L'))

for name, img in [('binary1.bmp', Im_bin1), ('binary3.bmp', Im_bin3)]:
    e = euler_number(img)
    print(f'{name}: Liczba Eulera = {e}')

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(img, cmap='gray')
    ax.set_title(f'{name} — Euler = {e}')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

### Wnioski
Liczba Eulera wyznacza topologiczną charakterystykę obrazu binarnego: E = obiekty − dziury.

Metoda bit-quad jest efektywna — wymaga jedynie jednego przejścia przez obraz z oknem 2×2.

Wynik zależy od przyjętej zasady spójności (4- vs 8-spójność).

## Zadanie 5. (opcjonalne)
Przetwarzanie obrazu monochromatycznego operatorem 3×3:

$$K = \begin{bmatrix} -1/9 & -1/9 & -1/9 \\ -1/9 & 8/9 & -1/9 \\ -1/9 & -1/9 & -1/9 \end{bmatrix}$$

In [ ]:
K = np.array([[-1/9, -1/9, -1/9],
              [-1/9,  8/9, -1/9],
              [-1/9, -1/9, -1/9]])

for name, img in [('B.bmp', Im_B), ('C.bmp', Im_C)]:
    result = filter2d(img, K)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(img, cmap='gray')
    axes[0].set_title(f'{name} — oryginał')
    axes[0].axis('off')
    axes[1].imshow(normalize(result), cmap='gray')
    axes[1].set_title(f'{name} — po filtracji')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

### Wnioski
Operator ten to filtr górnoprzepustowy (high-pass).

Macierz $K = I - \frac{1}{9}\mathbf{1}$ oznacza: wartość piksela minus średnia z otoczenia.

Efektem jest wydobycie krawędzi i szczegółów przy jednoczesnym usunięciu jednolitych (niskoczetstotliwościowych) obszarów obrazu.